# Business Impact for Fraud Detection

This notebook estimates the potential financial impact of the fraud detection model.

By identifying fraudulent transactions before they are completed, the model could help financial institutions prevent significant monetary losses.

In [1]:
import pandas as pd
import joblib

In [2]:
df = pd.read_csv("../data/raw/creditcard.csv")

In [3]:
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

In [4]:
model = joblib.load("../models/xgboost_fraud_model.pkl")

In [5]:
y_pred = model.predict(X_test)

In [6]:
fraud_amount = df[df["Class"]==1]["Amount"].mean()

frauds_detected = ((y_pred == 1) & (y_test == 1)).sum()

estimated_savings = frauds_detected * fraud_amount

print("Average fraud transaction:", fraud_amount)
print("Frauds detected:", frauds_detected)
print("Estimated fraud prevented ($):", estimated_savings)

Average fraud transaction: 122.21132113821139
Frauds detected: 82
Estimated fraud prevented ($): 10021.328333333335


The model was evaluated on the test dataset (20% of the data). Within this subset, the fraud detection system successfully identified fraudulent transactions representing approximately $10,000 in prevented losses.

Extrapolating this performance to the full dataset suggests that a large portion of the $60,000 total fraud volume could potentially be prevented.

In [7]:
from sklearn.metrics import recall_score

fraud_recall = recall_score(y_test, y_pred)

print("Fraud Recall:", fraud_recall)

Fraud Recall: 0.8367346938775511


### Interpretation

The model successfully identifies a significant proportion of fraudulent transactions.

By detecting these transactions in advance, the system could potentially prevent financial losses equivalent to the estimated savings calculated above.

In [8]:
y_prob = model.predict_proba(X_test)[:,1]

In [9]:
fraud_score = (1 - y_prob) * 1000

In [10]:
scores_df = pd.DataFrame({
    "fraud_probability": y_prob,
    "fraud_score": fraud_score
})

scores_df.head()

,fraud_probability,fraud_score
0,0.000027,999.973145
1,0.000017,999.982788
2,0.000202,999.797607
3,0.000020,999.979797
4,0.000183,999.817444


In [11]:
scores_df["risk_level"] = pd.cut(
    scores_df["fraud_score"],
    bins=[0,400,700,1000],
    labels=["High Risk","Medium Risk","Low Risk"]
)

In [12]:
scores_df["risk_level"].value_counts()

risk_level
Low Risk       56859
High Risk         94
Medium Risk        9
Name: count, dtype: int64

In [13]:
 total_fraud = df[df["Class"]==1]["Amount"].sum()

print("Total fraud volume:", total_fraud)

Total fraud volume: 60127.97


In [14]:
total_fraud = df[df["Class"]==1]["Amount"].sum()

estimated_prevented_full = total_fraud * fraud_recall

print("Estimated prevented fraud (full dataset):", estimated_prevented_full)

Estimated prevented fraud (full dataset): 50311.158571428576
